# Expansion-Aware Autorouting

This notebook demonstrates the hot-line autorouting workflow: direct grid candidates, generated U-loop candidates, reserved loop envelopes, solver-study export, and report review metadata.

## Visualization Approach

The most useful review surface is an interactive 3D scene, not a static 2D plot. The scene should show:

- selected route as a solid pipe centerline tube;
- alternate candidates as translucent tubes;
- obstacles and keepouts as transparent solids;
- expansion-loop reserved envelope as a transparent box;
- endpoints and route metadata next to the Markdown/JSON report.

This makes the engineering question visible: did the selected hot-line route reserve enough space for thermal movement while avoiding equipment and nearby lines?

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "tuba").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
from tuba import Model
from tuba.routing import (
    AutoroutingAgent,
    ExpansionAwareRouter,
    ExpansionLoopGenerator,
    ExpansionLoopSpec,
    GridRouter,
    SolverAcceptanceCriteria,
    ThermalRouteRequirement,
)
from tuba.routing.solver_loop import SolverLoopConfig
from tuba.routing.types import PipeRouteRequest, RouteEndpoint, RoutingConstraints, RoutingGridSpec


def build_hot_line_model() -> Model:
    model = Model("NotebookHotLineExpansionLoop")
    model.add_material(
        "steel",
        E=210e9,
        nu=0.3,
        rho=7850.0,
        alpha=12e-6,
        allowable_stress={20.0: 140e6, 180.0: 120e6},
    )
    model.add_pipe_section("DN80", OD=0.0889, WT=0.00549)
    model.define_load_case("Hot", gravity=True, pressure=1.2e6, temperature=180.0)
    model.add_obstacle(
        id="hot_equipment",
        type="cuboid",
        min_point=(3.4, -0.35, -0.35),
        max_point=(4.6, 0.35, 0.35),
    )
    return model


def build_hot_line_request() -> PipeRouteRequest:
    return PipeRouteRequest(
        id="HOT-NB-100",
        start=RouteEndpoint("PumpDischarge", (0.0, 0.0, 0.0), direction=(1.0, 0.0, 0.0)),
        goal=RouteEndpoint("RackTieIn", (8.0, 0.0, 0.0), direction=(1.0, 0.0, 0.0)),
        section="DN80",
        material="steel",
        constraints=RoutingConstraints(
            clearance=0.10,
            insulation_thickness=0.05,
            min_bend_radius=0.25,
        ),
        thermal_requirements=ThermalRouteRequirement(
            design_temperature_c=180.0,
            reference_temperature_c=20.0,
            line_length_m=8.0,
            thermal_expansion_coefficient=12e-6,
            metadata={"service": "hot oil"},
        ),
        solver_acceptance=SolverAcceptanceCriteria.hot_line_defaults(),
    )


model = build_hot_line_model()
request = build_hot_line_request()
request

In [ ]:
router = ExpansionAwareRouter(
    base_router=GridRouter(
        RoutingGridSpec(cell_size=0.5, margin=1.5),
        candidate_count=1,
    ),
    loop_generator=ExpansionLoopGenerator(
        loop_specs=(
            ExpansionLoopSpec(
                family="u_loop",
                width_m=2.0,
                depth_m=0.8,
                plane="xy",
                min_clearance_m=0.15,
            ),
        ),
    ),
)

agent = AutoroutingAgent(
    router=router,
    solver_config=SolverLoopConfig(
        run_solver=False,
        export_study=True,
        max_solver_candidates=2,
        load_case="Hot",
    ),
    output_root=PROJECT_ROOT / "routing_reports",
)

run = agent.route_pipe(model, request, apply=False)
selected = run.result.selected
selected.metadata if selected is not None else None

In [ ]:
candidate_rows = []
for idx, candidate in enumerate(run.result.candidates):
    candidate_rows.append(
        {
            "index": idx,
            "family": candidate.metadata.get("route_family", "unknown"),
            "valid": candidate.is_valid,
            "cost": round(candidate.cost, 3),
            "length": round(candidate.cost_breakdown.get("length", 0.0), 3),
            "bends": candidate.cost_breakdown.get("bends", 0.0),
            "has_reserved_envelope": "reserved_envelope" in candidate.metadata,
            "diagnostics": "; ".join(candidate.diagnostics),
        }
    )

candidate_rows

In [ ]:
print(f"Selected family: {selected.metadata.get('route_family') if selected else None}")
print(f"Report: {run.report_path}")
print(f"Study directory: {selected.metadata.get('solver', {}).get('study_dir') if selected else None}")

## Interactive 3D Review

The next cell exports a standalone HTML scene and then opens the same PyVista scene in the notebook when optional notebook visualization dependencies are installed. The transparent envelope is the space that later network routes should avoid.

In [ ]:
from IPython.display import Markdown, display
from tuba.routing.visualization import export_route_scene_html, show_route_scene

scene_path = PROJECT_ROOT / "routing_reports" / request.id / "route_scene.html"

try:
    exported_scene = export_route_scene_html(model, scene_path, request=request, result=run.result)
    display(Markdown(f"Standalone scene: `{exported_scene}`"))
    show_route_scene(model, request=request, result=run.result)
except ImportError as exc:
    display(Markdown(f"PyVista notebook visualization is not installed: `{exc}`"))
except Exception as exc:
    display(Markdown(f"Scene export/display failed in this environment: `{exc}`"))

## What To Review

- Does the selected U-loop reserve enough volume around the offset leg?
- Is the grid detour cheaper only because it ignores thermal flexibility, or is the loop actually lower cost?
- Do exported Code_Aster study files exist for the reviewed candidates?
- If solver execution is enabled later, do expansion ratio, sustained ratio, and anchor reaction stay inside the project limits?
- Do nearby network routes intersect the selected route's reserved envelope?